# LoRA 알아보기
LoRA는 Low-Rank Adaption으로 대규모 AI 모델을 적은 GPU 메모리와 적은 학습 비용으로 파인튜닝 하기 위한 기술입니다. 

특히 LLM을 특정 업무나 도메인에 맞게 학습시키기 위해 사용합니다.

일반적인 파인튜닝은 모델의 수많은 가중치를 직접 수정해야하지만 LoRA의 경우에는 원본 가중치는 그대로 살려두며 새로운 가중치 행렬을 학습시키는 방식을 사용합니다.

### 사용 모델
모델은 Qwen2.5-0.5B-Instruct를 사용하며 LoRA와 SFT를 사용합니다.

### 튜닝 목표
한국어 고객센터 응답을 `[안내]`/`[다음 단계]` 형식으로 만드는 LoRA 어댑터를 학습시킵니다.

### 새로 알게된 라이브러리
- accelerate: 학습 하드웨어 최적화를 위한 GPU활용, 분산 학습, 혼합 정밀도
- peft: 효율적인 파인튜닝을 위한 사용 -> LoRA/QLoRA을 사용할 수 있음
- trl: LLM 학습 및 정렬 가능 -> SFT, DPO, 보상 모델 학습 등에 사용 가능

In [1]:
import gc
from pathlib import Path
from typing import Literal, TypedDict

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BatchEncoding,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

# 타입 힌트: 변수명: 타입, 함수의 -> 타입은 반환값의 자료형입니다.
# 타입 힌트는 설명과 편집기 검사용이며, 값을 변환하거나 실행 중 타입을 강제하지 않습니다.
# list[T]는 T를 담는 리스트, A | B는 A 또는 B 타입을 뜻합니다.
TrainingExample = tuple[str, str, str]  # (질문, 안내 문구, 다음 단계)


class ChatMessage(TypedDict):
    """메시지 딕셔너리의 키와 값 타입. 실제 값은 일반 dict입니다."""

    role: Literal["system", "user", "assistant"]  # 이 세 문자열 중 하나
    content: str


class TrainingRecord(TypedDict):
    """학습 샘플 하나: 입력 메시지 목록과 정답 메시지 목록."""

    prompt: list[ChatMessage]
    completion: list[ChatMessage]

In [2]:
MODEL_ID: str = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_DIR: Path = Path("./qwen25-mokashop-lora")

# 테스트 
TEST_QUESTION: str = "배송지를 잘못 입력했어요. 어떻게 변경하나요?"


SYSTEM_PROMPT: str = (
  "당신은 가상의 온라인 쇼핑몰 '모카샵'의 고객지원 담당입니다. "
  "정중하고 정확하게 안내하세요. 주문 정보나 운영 정책을 임의로 만들어내지 마세요"
)

set_seed(42)
if not torch.cuda.is_available():
  raise RuntimeError("쿠다가 없어요 ㅠㅠ")

DTYPE: torch.dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0))
print("학습 dtype:", DTYPE)

GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
학습 dtype: torch.bfloat16


# GPT가 만들어준 학습 예시 데이터

In [ ]:
# (질문, 안내 문구, 다음 단계) 순서
examples: list[TrainingExample] = [
    ("배송 상태는 어디에서 확인하나요?", "주문 내역에서 배송 상태를 확인할 수 있습니다.", "로그인 후 주문 내역을 열어 주세요."),
    ("상품이 아직 도착하지 않았어요.", "배송이 예상보다 늦어질 수 있습니다. 현재 상태를 먼저 확인해 주세요.", "주문 내역의 배송 조회를 확인해 주세요."),
    ("주문을 취소하고 싶어요.", "취소 가능 여부는 주문 진행 단계에 따라 달라질 수 있습니다.", "주문 내역에서 취소 가능 상태인지 확인해 주세요."),
    ("결제 영수증은 어디서 보나요?", "결제 관련 문서는 주문 내역에서 확인할 수 있습니다.", "주문 상세 화면에서 영수증 메뉴를 찾아 주세요."),
    ("반품은 어떻게 신청하나요?", "반품 신청 가능 여부는 주문 상태와 상품 조건에 따라 다를 수 있습니다.", "주문 내역에서 반품 신청 메뉴를 확인해 주세요."),
    ("상품을 다른 옵션으로 바꾸고 싶어요.", "옵션 변경 가능 여부는 주문 처리 상황에 따라 달라질 수 있습니다.", "주문 상태를 확인한 뒤 고객지원에 변경 가능 여부를 문의해 주세요."),
    ("주소가 바뀌었는데 배송지 수정이 가능할까요?", "배송지 변경 가능 여부는 상품 발송 상태에 따라 달라질 수 있습니다.", "주문 내역에서 발송 상태를 확인하고 고객지원에 문의해 주세요."),
    ("쿠폰이 적용되지 않아요.", "쿠폰마다 사용 조건과 적용 가능 상품이 다를 수 있습니다.", "쿠폰의 유효 기간과 사용 조건을 먼저 확인해 주세요."),
    ("상품이 파손된 상태로 도착했어요.", "배송 중 상품이 손상되었다면 상태 확인이 필요합니다.", "상품과 포장 상태를 촬영한 뒤 고객지원에 문의해 주세요."),
    ("잘못된 상품을 받았어요.", "주문한 상품과 수령한 상품이 다른 경우 확인이 필요합니다.", "주문 내역과 받은 상품 사진을 준비해 고객지원에 문의해 주세요."),
    ("재입고 소식을 받을 수 있나요?", "재입고 알림 기능은 상품에 따라 제공 여부가 다를 수 있습니다.", "상품 상세 페이지에서 알림 옵션이 있는지 확인해 주세요."),
    ("회원 정보를 수정하려면 어떻게 하나요?", "일부 회원 정보는 계정 설정에서 변경할 수 있습니다.", "로그인 후 계정 설정 메뉴를 확인해 주세요."),
    ("비밀번호를 잊어버렸어요.", "로그인 화면에서 비밀번호 재설정을 진행할 수 있습니다.", "비밀번호 찾기 메뉴에서 본인 확인 절차를 진행해 주세요."),
    ("결제 수단을 변경하고 싶어요.", "결제 수단 변경 가능 여부는 결제 완료 여부에 따라 달라질 수 있습니다.", "주문 상태를 확인한 후 고객지원에 가능한 방법을 문의해 주세요."),
    ("배송비가 왜 붙었나요?", "배송비는 주문 조건과 배송 지역 등에 따라 달라질 수 있습니다.", "결제 화면의 배송비 안내와 주문 조건을 확인해 주세요."),
    ("선물 포장을 할 수 있나요?", "선물 포장 지원 여부는 상품이나 주문 옵션에 따라 다를 수 있습니다.", "상품 상세 페이지 또는 주문 옵션을 확인해 주세요."),
    ("주문 번호를 찾을 수 없어요.", "주문 번호는 일반적으로 주문 내역이나 주문 확인 안내에서 찾을 수 있습니다.", "로그인 후 주문 내역을 먼저 확인해 주세요."),
    ("품절된 상품은 언제 다시 판매하나요?", "재입고 일정은 상품별로 달라 확정된 날짜를 안내하기 어렵습니다.", "상품 상세 페이지의 재입고 공지를 확인해 주세요."),
    # --- 여기까지 18개 --- #
    # 검증용: 학습에 포함하지 않는 예시
    ("로그아웃은 어디에서 하나요?", "로그아웃 메뉴는 계정 또는 프로필 메뉴에 있을 수 있습니다.", "화면의 계정 메뉴를 열어 로그아웃 항목을 확인해 주세요."),
    ("휴대폰 번호를 바꾸고 싶습니다.", "연락처 수정 가능 여부는 계정 설정에서 확인할 수 있습니다.", "계정 설정의 개인정보 수정 항목을 확인해 주세요."),
    ("상품 상세 정보는 어디서 확인하나요?", "상품의 옵션과 사양은 상품 상세 페이지에서 확인할 수 있습니다.", "상품 상세 페이지의 설명과 옵션을 살펴봐 주세요."),
    ("배송 조회 버튼이 없어요.", "배송 조회 기능은 상품 발송 상태에 따라 표시 여부가 달라질 수 있습니다.", "주문 상세 화면에서 발송 상태를 확인하고 필요하면 고객지원에 문의해 주세요."),
    # --- 검증 데이터 4개 --- #
]

# 위의 데이터 하나를 받아 3데이터셋으로 분리하여 작업
def to_training_record(item: TrainingExample) -> TrainingRecord:
    """문자열 3개로 된 튜플을 prompt/completion 딕셔너리로 변환합니다."""
    question, notice, action = item
    return {
        # Qwen 채팅 템플릿은 TRL SFTTrainer가 자동으로 적용합니다.
        # TRL: Transformemr Reinforcement Learning으로 LLM 학습 라이브러리
        # SFT: Supervised Fine-Tuning로 질문과 정답 데이터를 이용한 지도 파인튜닝을 하는 라이브러리
        # Trainer: 학습 과정을 자동으로 관리해주는 클래스
        # SFTTrainer: 학습 데이터를 토큰으로 변환하여 모델의 예측과 정답을 비교하여 Loss 계산 후 가중치 업데이트
        # └━ 로그까지 자동으로 기록해줌
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        # prompt가 아닌 completion(정답)에 대해서만 손실을 계산합니다.
        "completion": [
            {
                # 모델이 학습할 정답 형태입니다.
                "role": "assistant",
                "content": f"[안내]\n{notice}\n\n[다음 단계]\n{action}",
            }
        ],
    }

# 학습하기 위한 데이터를 0~17와 검증용 18~21을 뽑아서 학습 레코드 (dict)로 만들어 Dataset으로 만들기.
train_ds: Dataset = Dataset.from_list([to_training_record(x) for x in examples[:18]])
eval_ds: Dataset = Dataset.from_list([to_training_record(x) for x in examples[18:]])
# Dataset은 __len__을 제공하고, 인덱싱을 통해 피처를 뽑을 수 있음
print(f"학습 {len(train_ds)}개, 검증 {len(eval_ds)}개")
print("정답 예시:", train_ds[0]["completion"][0]["content"])

학습 18개, 검증 4개
정답 예시: [안내]
주문 내역에서 배송 상태를 확인할 수 있습니다.

[다음 단계]
로그인 후 주문 내역을 열어 주세요.


In [4]:
tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(MODEL_ID)
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
model.to("cuda")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

c:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hurwa\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

# 생성 함수 정의 및 학습 전 출력 확인

In [30]:
@torch.inference_mode()
def generate_answer(
    current_model: PreTrainedModel | PeftModel, 
    question: str
  ) -> str:
  """기본 모델 또는 LoRA 모델로 답변 문자열을 생성합니다."""
  # 모델을 평가(추론) 모드로 전환
  current_model.eval()

  messages: list[ChatMessage] = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question}
  ]

  # BatchEncoding: input_ids, attention_mask 등을 담는 딕셔너리 형태의 객체
  encoded_inputs: BatchEncoding = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
  )

  # return_tensors="pt"이므로 각 값은 torch.Tensor입니다.
  print(type(encoded_inputs["input_ids"]))

  # print(encoded_inputs)
  device: torch.device = next(current_model.parameters()).device

  # GPU로 옮긴 텐서 사전. 각 텐서의 모양은 [배치 크기, 입력 토큰 수]입니다.
  inputs: dict[str, torch.Tensor] = {k: v.to(device) for k, v in encoded_inputs.items()}
  
  # input_ids와 attention_mask를 
  # for key, value in inputs.items():
  #   print(key, value.shape)
  print(list((k, (v.shape if hasattr(v, "shape") else None)) for k, v in inputs.items()))

  # 기본 반환값은 입력 + 생성 토큰의 정수 텐서: [배치 크기, 전체 토큰 수]
  outputs: torch.Tensor = current_model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
  )

  # 첫 번째 답변에서 입력 부분을 제외한 1차원 텐서: [생성 토큰 수]
  generated_tokens: torch.Tensor = outputs[0, inputs["input_ids"].shape[1]:]
  return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("=== 학습 전 ===")
print(generate_answer(model, TEST_QUESTION))

=== 학습 전 ===
<class 'torch.Tensor'>
[('input_ids', torch.Size([1, 82])), ('attention_mask', torch.Size([1, 82]))]
죄송합니다, 배송지를 잘못 입력한 것 같습니다. 이 문제를 해결하기 위해 다음 단계를 따르시면 좋습니다:

1. **주문 확인**: 먼저, 원하시는 상품과 수량을 확인해 보세요.

2. **배송지 확인**: 배송지를 확인하시고, 해당 배송지에 대한 정보를 확인하세요.

3. **변경 요청**: 배송지를 잘못 입력한 부분에 대해 변경할 수 있는 방법을 알려주시면, 이를 통해 배송지를 다시 설정할 수 있습니다.

4. **결제


# LoRA 설정

In [31]:
lora_config: LoraConfig = LoraConfig(
  task_type=TaskType.CAUSAL_LM,
  r=8,              # 저랭크 차원
  lora_alpha=16,    # loRA 변화량 스케일 관련 값
  lora_dropout=0.05,
  target_modules=["q_proj", "v_proj"], # Qwen2.5의 attention projection
  bias="none",
)

In [33]:
for name, layer in model.named_modules():
  if name.endswith(("q_proj", "v_proj", "k_proj")):
    if ".layers.0" in name:
      print(name, layer.weight.shape)

model.layers.0.self_attn.q_proj torch.Size([896, 896])
model.layers.0.self_attn.k_proj torch.Size([128, 896])
model.layers.0.self_attn.v_proj torch.Size([128, 896])


# SFTTrainer로 LoRA 파인튜닝

In [34]:
training_args: SFTConfig = SFTConfig(
  output_dir="./qwen25-mokashop-runs",
  per_device_train_batch_size=1,
  per_device_eval_batch_size=1,
  gradient_accumulation_steps=2,
  num_train_epochs=3,
  learning_rate=2e-4,
  max_length=256,
  completion_only_loss=True,
  eval_strategy="epoch",
  save_strategy="no",
  logging_steps=2,
  gradient_checkpointing=False,
    bf16=(DTYPE == torch.bfloat16),
    fp16=(DTYPE == torch.float16),
    optim="adamw_torch",
    report_to="none",
    seed=42,
)

trainer: SFTTrainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()
trainer.train()
print("최종 검증:", trainer.evaluate())

Tokenizing train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.200920,2.068691,2.042904,0.540621,2327.000000
2,1.759421,1.847971,1.997276,0.579194,4654.000000
3,1.777690,1.768198,1.959632,0.607441,6981.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
1.777690,1.768198,3,1.959632,0.607441,6981.000000


최종 검증: {'eval_loss': 1.7681975364685059, 'eval_entropy': 1.9596319496631622, 'eval_mean_token_accuracy': 0.6074410080909729, 'eval_num_tokens': 6981.0}


# 셀 7 학습 후 추론: 학습 전과 동일 질문 비교

In [35]:
print("=== LoRA 학습 후 ===")
print(generate_answer(trainer.model, TEST_QUESTION))

=== LoRA 학습 후 ===
<class 'torch.Tensor'>
[('input_ids', torch.Size([1, 82])), ('attention_mask', torch.Size([1, 82]))]
죄송합니다, 배송지에 대한 정보를 잘못 입력했습니다. 이 문제를 해결하기 위해 다음과 같은 방법들을 확인해보시는 것이 좋습니다:

1. **주문 상세 페이지 확인**: 
   - 주문 내역에서 배송지에 대한 정보가 제대로 입력되지 않았음을 확인할 수 있습니다.

2. **주문 내역 확인**:
   - 주문 내역에서 배송지에 대한 정보가 제대로 입력되지 않았음을 확인할 수 있습니다.

3. **주문 내역 수정**:
   - 주문 내


# 셀 8 LoRA 어댑터 + 토크나이저 저장

In [36]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)   # 원본 모델 전체가 아닌 LoRA 어댑터 저장
tokenizer.save_pretrained(ADAPTER_DIR)
print("저장 위치:", ADAPTER_DIR.resolve())

저장 위치: C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\02_self_study\machine_learning\LoRA\qwen25-mokashop-lora


# 셀 9 저장된 어댑터를 다시 불러와 추론

In [37]:
# 학습 객체를 정리하여 GPU 메모리 사용을 줄입니다.
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

base_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
base_model.to("cuda")
reloaded_model: PeftModel = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
reloaded_model.eval()

print("=== 어댑터 재로딩 후 ===")
print(generate_answer(reloaded_model, "반품을 요청하려면 무엇부터 해야 하나요?"))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== 어댑터 재로딩 후 ===
<class 'torch.Tensor'>
[('input_ids', torch.Size([1, 82])), ('attention_mask', torch.Size([1, 82]))]
주문 확인 후, 고객센터에 문의해주세요.
